# De l'architecture Transformer à l'exploitation en production
## Un parcours complet : pré-entraînement → fine-tuning/RLHF → inférence & infrastructure

> Ce notebook retrace les grandes étapes de construction et d'exploitation d'un grand modèle de langage (LLM),
> en partant des fondations architecturales jusqu'aux enjeux d'infrastructure en production.
> Chaque section contextualise le concept avant de l'illustrer.

---


---
# Partie 1 — Architectures de modèles de langage

Avant de parler d'entraînement, il faut comprendre les trois familles architecturales qui existent dans la bibliothèque Transformers.
Elles partagent toutes la même brique de base — le Transformer — mais diffèrent dans la façon dont elles lisent et produisent du texte.


In [ ]:
from IPython.display import HTML
HTML('''
<svg width="100%" viewBox="0 0 700 320" xmlns="http://www.w3.org/2000/svg">
<style>svg{font-family:system-ui,sans-serif;} .t{font-size:14px;fill:#1a1a1a;} .ts{font-size:12px;fill:#555;} .th{font-size:14px;font-weight:600;fill:#1a1a1a;}</style>
<defs><marker id="arr" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/></marker></defs>

<!-- Encoder-only -->
<rect x="30" y="50" width="190" height="200" rx="10" fill="#E6F1FB" stroke="#185FA5" stroke-width="1"/>
<text class="th" x="125" y="78" text-anchor="middle">Encodeur seul</text>
<text class="ts" x="125" y="96" text-anchor="middle">ex: BERT</text>
<rect x="50" y="110" width="150" height="34" rx="6" fill="#185FA5" opacity=".15" stroke="#185FA5" stroke-width=".5"/>
<text class="ts" x="125" y="132" text-anchor="middle">Attention bidirectionnelle</text>
<rect x="50" y="152" width="150" height="34" rx="6" fill="#185FA5" opacity=".15" stroke="#185FA5" stroke-width=".5"/>
<text class="ts" x="125" y="174" text-anchor="middle">Lit dans les 2 sens</text>
<text class="ts" x="125" y="220" text-anchor="middle" fill="#185FA5">Classification, NER</text>
<text class="ts" x="125" y="238" text-anchor="middle" fill="#185FA5">Question answering</text>

<!-- Decoder-only -->
<rect x="255" y="50" width="190" height="200" rx="10" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1"/>
<text class="th" x="350" y="78" text-anchor="middle">Décodeur seul</text>
<text class="ts" x="350" y="96" text-anchor="middle">ex: GPT, Llama</text>
<rect x="275" y="110" width="150" height="34" rx="6" fill="#0F6E56" opacity=".15" stroke="#0F6E56" stroke-width=".5"/>
<text class="ts" x="350" y="132" text-anchor="middle">Attention causale (gauche→droite)</text>
<rect x="275" y="152" width="150" height="34" rx="6" fill="#0F6E56" opacity=".15" stroke="#0F6E56" stroke-width=".5"/>
<text class="ts" x="350" y="174" text-anchor="middle">Génération autorégressive</text>
<text class="ts" x="350" y="220" text-anchor="middle" fill="#0F6E56">Génération de texte</text>
<text class="ts" x="350" y="238" text-anchor="middle" fill="#0F6E56">Code, conversation</text>

<!-- Encoder-Decoder -->
<rect x="480" y="50" width="190" height="200" rx="10" fill="#FAEEDA" stroke="#BA7517" stroke-width="1"/>
<text class="th" x="575" y="78" text-anchor="middle">Encodeur-Décodeur</text>
<text class="ts" x="575" y="96" text-anchor="middle">ex: T5, BART</text>
<rect x="500" y="110" width="150" height="34" rx="6" fill="#BA7517" opacity=".15" stroke="#BA7517" stroke-width=".5"/>
<text class="ts" x="575" y="132" text-anchor="middle">Comprend l'entrée</text>
<rect x="500" y="152" width="150" height="34" rx="6" fill="#BA7517" opacity=".15" stroke="#BA7517" stroke-width=".5"/>
<text class="ts" x="575" y="174" text-anchor="middle">Génère la sortie</text>
<text class="ts" x="575" y="220" text-anchor="middle" fill="#BA7517">Traduction, résumé</text>
<text class="ts" x="575" y="238" text-anchor="middle" fill="#BA7517">Tâches seq2seq</text>

<text class="ts" x="350" y="290" text-anchor="middle" fill="#888">Les trois familles partagent la même brique Transformer — elles diffèrent par leur masque d'attention et leur objectif d'entraînement.</text>
</svg>
''')

## 1.1 — Tâches séquence → séquence

Une tâche **seq2seq** prend une suite de tokens en entrée et produit une autre suite de longueur **variable** en sortie.
C'est fondamentalement différent d'une classification qui produit un label fixe.


In [ ]:
from IPython.display import HTML
HTML('''
<svg width="100%" viewBox="0 0 680 200" xmlns="http://www.w3.org/2000/svg">
<style>svg{font-family:system-ui,sans-serif;} .ts{font-size:12px;fill:#555;} .th{font-size:13px;font-weight:600;fill:#1a1a1a;}</style>
<defs><marker id="arr" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/></marker></defs>

<!-- Traduction -->
<text class="th" x="40" y="30">Traduction — 3 tokens → 3 tokens</text>
<rect x="40" y="42" width="56" height="28" rx="5" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="68" y="61" text-anchor="middle">"The"</text>
<rect x="104" y="42" width="56" height="28" rx="5" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="132" y="61" text-anchor="middle">"cat"</text>
<rect x="168" y="42" width="66" height="28" rx="5" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="201" y="61" text-anchor="middle">"sleeps"</text>
<line x1="238" y1="56" x2="268" y2="56" stroke="#888" stroke-width="1.2" marker-end="url(#arr)"/>
<rect x="270" y="38" width="72" height="36" rx="6" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1"/><text class="ts" x="306" y="61" text-anchor="middle" font-weight="600">Modèle</text>
<line x1="342" y1="56" x2="372" y2="56" stroke="#888" stroke-width="1.2" marker-end="url(#arr)"/>
<rect x="374" y="42" width="46" height="28" rx="5" fill="#FAECE7" stroke="#993C1D" stroke-width=".8"/><text class="ts" x="397" y="61" text-anchor="middle">"Le"</text>
<rect x="428" y="42" width="56" height="28" rx="5" fill="#FAECE7" stroke="#993C1D" stroke-width=".8"/><text class="ts" x="456" y="61" text-anchor="middle">"chat"</text>
<rect x="492" y="42" width="52" height="28" rx="5" fill="#FAECE7" stroke="#993C1D" stroke-width=".8"/><text class="ts" x="518" y="61" text-anchor="middle">"dort"</text>

<!-- Résumé -->
<text class="th" x="40" y="118">Résumé — 4 tokens → 2 tokens (sortie plus courte !)</text>
<rect x="40" y="130" width="46" height="28" rx="5" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="63" y="149" text-anchor="middle">"La"</text>
<rect x="94" y="130" width="66" height="28" rx="5" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="127" y="149" text-anchor="middle">"voiture"</text>
<rect x="168" y="130" width="60" height="28" rx="5" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="198" y="149" text-anchor="middle">"rouge"</text>
<rect x="236" y="130" width="52" height="28" rx="5" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="262" y="149" text-anchor="middle">"est..."</text>
<line x1="292" y1="144" x2="318" y2="144" stroke="#888" stroke-width="1.2" marker-end="url(#arr)"/>
<rect x="320" y="130" width="72" height="28" rx="6" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1"/><text class="ts" x="356" y="149" text-anchor="middle" font-weight="600">Modèle</text>
<line x1="394" y1="144" x2="420" y2="144" stroke="#888" stroke-width="1.2" marker-end="url(#arr)"/>
<rect x="422" y="130" width="66" height="28" rx="5" fill="#FAECE7" stroke="#993C1D" stroke-width=".8"/><text class="ts" x="455" y="149" text-anchor="middle">"Voiture"</text>
<rect x="496" y="130" width="60" height="28" rx="5" fill="#FAECE7" stroke="#993C1D" stroke-width=".8"/><text class="ts" x="526" y="149" text-anchor="middle">"rouge"</text>
<text class="ts" x="580" y="149" fill="#993C1D" font-weight="600">← plus court !</text>
</svg>
''')

---
# Partie 2 — Pré-entraînement

Le pré-entraînement est la phase la plus coûteuse et la plus fondamentale. Le modèle apprend à prédire
le token suivant sur des milliards de textes issus d'internet, de livres, de code...
Il ne sait encore pas "répondre à des questions" — il sait continuer du texte.

> **Coût estimé pour GPT-4 :** ~100 millions de dollars, des semaines sur des milliers de GPU.


## 2.1 — Tokenisation : Byte Pair Encoding (BPE)

Avant d'entraîner le modèle, il faut convertir le texte en tokens.
Le BPE part des caractères individuels et fusionne itérativement les paires les plus fréquentes.


In [ ]:
from IPython.display import HTML
HTML('''
<svg width="100%" viewBox="0 0 680 320" xmlns="http://www.w3.org/2000/svg">
<style>svg{font-family:system-ui,sans-serif;} .ts{font-size:12px;fill:#555;} .th{font-size:13px;font-weight:600;fill:#1a1a1a;} .box{fill:#F1EFE8;stroke:#5F5E5A;stroke-width:.5;}</style>
<defs><marker id="arr" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/></marker></defs>

<text class="th" x="40" y="28">Corpus initial (caractères séparés)</text>
<rect x="40" y="36" width="600" height="30" rx="6" class="box"/>
<text class="ts" x="340" y="56" text-anchor="middle">l o w _  ,  l o w e r _  ,  n e w e s t _  ,  w i d e s t _</text>

<line x1="340" y1="66" x2="340" y2="90" stroke="#888" stroke-width="1" marker-end="url(#arr)"/>
<text class="ts" x="500" y="82" fill="#888">paire la plus fréquente : "e s"</text>

<text class="th" x="40" y="108">Itération 1 — fusion "e"+"s" → "es"</text>
<rect x="40" y="116" width="600" height="30" rx="6" class="box"/>
<text class="ts" x="340" y="136" text-anchor="middle">l o w _  ,  l o w e r _  ,  n <tspan font-weight="700" fill="#185FA5">es</tspan> t _  ,  w i d <tspan font-weight="700" fill="#185FA5">es</tspan> t _</text>

<line x1="340" y1="146" x2="340" y2="170" stroke="#888" stroke-width="1" marker-end="url(#arr)"/>
<text class="ts" x="500" y="162" fill="#888">paire la plus fréquente : "es t"</text>

<text class="th" x="40" y="188">Itération 2 — fusion "es"+"t" → "est"</text>
<rect x="40" y="196" width="600" height="30" rx="6" class="box"/>
<text class="ts" x="340" y="216" text-anchor="middle">l o w _  ,  l o w e r _  ,  n <tspan font-weight="700" fill="#0F6E56">est</tspan> _  ,  w i d <tspan font-weight="700" fill="#0F6E56">est</tspan> _</text>

<line x1="340" y1="226" x2="340" y2="250" stroke="#888" stroke-width="1" marker-end="url(#arr)"/>
<text class="ts" x="500" y="242" fill="#888">paire la plus fréquente : "l o"</text>

<text class="th" x="40" y="268">Itération 3 — fusion "l"+"o" → "lo"</text>
<rect x="40" y="276" width="600" height="30" rx="6" class="box"/>
<text class="ts" x="340" y="296" text-anchor="middle"><tspan font-weight="700" fill="#BA7517">lo</tspan> w _  ,  <tspan font-weight="700" fill="#BA7517">lo</tspan> w e r _  ,  n est _  ,  w i d est _</text>
</svg>
''')

**Avantages du BPE :**
- Les mots fréquents deviennent un seul token (`"the"`, `"chat"`)
- Les mots rares sont décomposés en sous-unités connues (pas de token `[UNKNOWN]`)
- La casse est préservée car elle porte du sens : `"Apple"` ≠ `"apple"` (entreprise vs fruit)
- GPT-2 utilise un vocabulaire de **50 257 tokens**


## 2.2 — Architecture GPT-2 : décodeur autorégressif

GPT-2 est un modèle **décodeur seul**. Il génère du texte de gauche à droite, un token à la fois.
Chaque bloc décodeur contient une couche d'attention masquée et une couche feed-forward.


In [ ]:
from IPython.display import HTML
HTML('''
<svg width="100%" viewBox="0 0 680 380" xmlns="http://www.w3.org/2000/svg">
<style>svg{font-family:system-ui,sans-serif;} .ts{font-size:12px;fill:#555;} .th{font-size:13px;font-weight:600;fill:#1a1a1a;}</style>
<defs><marker id="arr" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/></marker></defs>

<!-- Input -->
<rect x="40" y="320" width="120" height="34" rx="6" fill="#E6F1FB" stroke="#185FA5" stroke-width="1"/>
<text class="ts" x="100" y="342" text-anchor="middle">Token embedding</text>
<rect x="180" y="320" width="140" height="34" rx="6" fill="#E6F1FB" stroke="#185FA5" stroke-width="1"/>
<text class="ts" x="250" y="342" text-anchor="middle">+ Positional encoding</text>

<line x1="200" y1="320" x2="200" y2="294" stroke="#888" stroke-width="1" marker-end="url(#arr)"/>

<!-- Decoder block -->
<rect x="80" y="160" width="240" height="126" rx="10" fill="#F1EFE8" stroke="#5F5E5A" stroke-width="1" stroke-dasharray="5 3"/>
<text class="ts" x="200" y="178" text-anchor="middle" fill="#888">Bloc décodeur × N</text>

<rect x="100" y="184" width="200" height="30" rx="6" fill="#FAEEDA" stroke="#BA7517" stroke-width=".8"/>
<text class="ts" x="200" y="204" text-anchor="middle">Masked Multi-Head Attention</text>

<rect x="100" y="222" width="200" height="30" rx="6" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".8"/>
<text class="ts" x="200" y="242" text-anchor="middle">Add &amp; Norm</text>

<rect x="100" y="258" width="200" height="22" rx="6" fill="#FAEEDA" stroke="#BA7517" stroke-width=".8"/>
<text class="ts" x="200" y="274" text-anchor="middle">Feed Forward</text>

<line x1="200" y1="160" x2="200" y2="134" stroke="#888" stroke-width="1" marker-end="url(#arr)"/>

<!-- LM Head -->
<rect x="100" y="100" width="200" height="30" rx="6" fill="#FAECE7" stroke="#993C1D" stroke-width="1"/>
<text class="ts" x="200" y="120" text-anchor="middle">LM Head — Linear (dim → 50 257)</text>

<line x1="200" y1="100" x2="200" y2="74" stroke="#888" stroke-width="1" marker-end="url(#arr)"/>

<rect x="100" y="44" width="200" height="28" rx="6" fill="#FCEBEB" stroke="#A32D2D" stroke-width="1"/>
<text class="ts" x="200" y="63" text-anchor="middle">Softmax → probabilités</text>

<!-- Annotations -->
<text class="ts" x="380" y="200" fill="#BA7517">← masque les tokens futurs</text>
<text class="ts" x="380" y="270" fill="#888">← projection non-linéaire</text>
<text class="ts" x="380" y="120" fill="#993C1D">← 1 score par token du vocabulaire</text>
<text class="ts" x="380" y="62" fill="#A32D2D">← distribution de proba (somme = 1)</text>

<!-- Output -->
<text class="ts" x="200" y="22" text-anchor="middle" fill="#185FA5" font-weight="600">→ Token prédit</text>
</svg>
''')

## 2.3 — Masked Self-Attention : le masque triangulaire

Le masque garantit que chaque token ne peut "voir" que les tokens qui le précèdent.
À l'**entraînement**, il empêche de tricher en regardant la réponse.
À l'**inférence**, les tokens futurs n'existent tout simplement pas encore — le masque est naturel.


In [ ]:
from IPython.display import HTML
HTML('''
<svg width="100%" viewBox="0 0 680 280" xmlns="http://www.w3.org/2000/svg">
<style>svg{font-family:system-ui,sans-serif;} .ts{font-size:11px;fill:#555;} .th{font-size:13px;font-weight:600;fill:#1a1a1a;}</style>

<text class="th" x="40" y="24">Matrice d'attention — triangulaire inférieure</text>
<text class="ts" x="40" y="42">Chaque ligne = un token qui "regarde". Chaque colonne = un token regardé.</text>

<!-- Labels colonnes -->
<text class="ts" x="175" y="66" text-anchor="middle">"Le"</text>
<text class="ts" x="225" y="66" text-anchor="middle">"chat"</text>
<text class="ts" x="275" y="66" text-anchor="middle">"dort"</text>
<text class="ts" x="325" y="66" text-anchor="middle">"sur"</text>

<!-- Row 1 -->
<text class="ts" x="150" y="92" text-anchor="end">"Le" prédit →</text>
<rect x="155" y="72" width="44" height="30" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".8"/><text class="ts" x="177" y="92" text-anchor="middle">1</text>
<rect x="205" y="72" width="44" height="30" rx="4" fill="#ddd" opacity=".5"/><text class="ts" x="227" y="92" text-anchor="middle" fill="#aaa">0</text>
<rect x="255" y="72" width="44" height="30" rx="4" fill="#ddd" opacity=".5"/><text class="ts" x="277" y="92" text-anchor="middle" fill="#aaa">0</text>
<rect x="305" y="72" width="44" height="30" rx="4" fill="#ddd" opacity=".5"/><text class="ts" x="327" y="92" text-anchor="middle" fill="#aaa">0</text>
<text class="ts" x="360" y="92" fill="#0F6E56">voit seulement lui-même</text>

<!-- Row 2 -->
<text class="ts" x="150" y="128" text-anchor="end">"chat" prédit →</text>
<rect x="155" y="108" width="44" height="30" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".8"/><text class="ts" x="177" y="128" text-anchor="middle">1</text>
<rect x="205" y="108" width="44" height="30" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".8"/><text class="ts" x="227" y="128" text-anchor="middle">1</text>
<rect x="255" y="108" width="44" height="30" rx="4" fill="#ddd" opacity=".5"/><text class="ts" x="277" y="128" text-anchor="middle" fill="#aaa">0</text>
<rect x="305" y="108" width="44" height="30" rx="4" fill="#ddd" opacity=".5"/><text class="ts" x="327" y="128" text-anchor="middle" fill="#aaa">0</text>
<text class="ts" x="360" y="128" fill="#0F6E56">voit "Le" + lui-même</text>

<!-- Row 3 -->
<text class="ts" x="150" y="164" text-anchor="end">"dort" prédit →</text>
<rect x="155" y="144" width="44" height="30" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".8"/><text class="ts" x="177" y="164" text-anchor="middle">1</text>
<rect x="205" y="144" width="44" height="30" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".8"/><text class="ts" x="227" y="164" text-anchor="middle">1</text>
<rect x="255" y="144" width="44" height="30" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".8"/><text class="ts" x="277" y="164" text-anchor="middle">1</text>
<rect x="305" y="144" width="44" height="30" rx="4" fill="#ddd" opacity=".5"/><text class="ts" x="327" y="164" text-anchor="middle" fill="#aaa">0</text>

<!-- Row 4 -->
<text class="ts" x="150" y="200" text-anchor="end">"sur" prédit →</text>
<rect x="155" y="180" width="44" height="30" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".8"/><text class="ts" x="177" y="200" text-anchor="middle">1</text>
<rect x="205" y="180" width="44" height="30" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".8"/><text class="ts" x="227" y="200" text-anchor="middle">1</text>
<rect x="255" y="180" width="44" height="30" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".8"/><text class="ts" x="277" y="200" text-anchor="middle">1</text>
<rect x="305" y="180" width="44" height="30" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".8"/><text class="ts" x="327" y="200" text-anchor="middle">1</text>
<text class="ts" x="360" y="200" fill="#0F6E56">voit tout</text>

<!-- Diagonal hint -->
<line x1="155" y1="72" x2="350" y2="210" stroke="#BA7517" stroke-width="1" stroke-dasharray="4 3" opacity=".6"/>

<rect x="40" y="230" width="600" height="36" rx="6" fill="#F1EFE8" stroke="#5F5E5A" stroke-width=".5"/>
<text class="ts" x="340" y="244" text-anchor="middle">À l'inférence : la matrice grandit d'une ligne/colonne à chaque token généré.</text>
<text class="ts" x="340" y="258" text-anchor="middle">Les 0 en haut à droite ne masquent rien — ces tokens n'existent pas encore.</text>
</svg>
''')

## 2.4 — LM Head : des hidden states aux probabilités

La dernière couche du modèle produit un vecteur de **50 257 logits** (un score par token du vocabulaire).
Le Softmax les convertit en probabilités. Le décalage (shift) à l'entraînement aligne prédictions et labels.

| Étape | Opération | Résultat |
|---|---|---|
| Hidden state | Vecteur de dim 768 | Représentation du token courant |
| LM Head (Linear) | 768 × 50 257 | 50 257 logits bruts |
| Softmax | exp(logit) / Σexp | 50 257 probabilités (somme = 1) |
| Sélection | Stratégie de décodage | 1 token prédit |


---
# Partie 3 — Du modèle fondationnel à l'assistant : Fine-tuning & RLHF

Le modèle fondationnel sait prédire des tokens — mais il ne sait pas "répondre".
Pour en faire un assistant, on passe par deux phases supplémentaires bien moins coûteuses.


In [ ]:
from IPython.display import HTML
HTML('''
<svg width="100%" viewBox="0 0 680 320" xmlns="http://www.w3.org/2000/svg">
<style>svg{font-family:system-ui,sans-serif;} .ts{font-size:12px;fill:#555;} .th{font-size:13px;font-weight:600;fill:#1a1a1a;}</style>
<defs><marker id="arr" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/></marker></defs>

<!-- Pre-training -->
<rect x="30" y="40" width="180" height="90" rx="10" fill="#E6F1FB" stroke="#185FA5" stroke-width="1.2"/>
<text class="th" x="120" y="64" text-anchor="middle">Pré-entraînement</text>
<text class="ts" x="120" y="84" text-anchor="middle">~100M$ | semaines</text>
<text class="ts" x="120" y="102" text-anchor="middle">Milliards de tokens</text>
<text class="ts" x="120" y="118" text-anchor="middle">Next token prediction</text>

<line x1="212" y1="85" x2="248" y2="85" stroke="#888" stroke-width="1.5" marker-end="url(#arr)"/>
<text class="ts" x="230" y="76" text-anchor="middle">modèle</text>
<text class="ts" x="230" y="98" text-anchor="middle">fondationnel</text>

<!-- SFT -->
<rect x="250" y="40" width="170" height="90" rx="10" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1.2"/>
<text class="th" x="335" y="64" text-anchor="middle">Fine-tuning (SFT)</text>
<text class="ts" x="335" y="84" text-anchor="middle">~quelques M$</text>
<text class="ts" x="335" y="102" text-anchor="middle">10k-100k exemples</text>
<text class="ts" x="335" y="118" text-anchor="middle">question → réponse</text>

<line x1="422" y1="85" x2="458" y2="85" stroke="#888" stroke-width="1.5" marker-end="url(#arr)"/>

<!-- RLHF -->
<rect x="460" y="40" width="190" height="90" rx="10" fill="#FAEEDA" stroke="#BA7517" stroke-width="1.2"/>
<text class="th" x="555" y="64" text-anchor="middle">RLHF</text>
<text class="ts" x="555" y="84" text-anchor="middle">~quelques M$</text>
<text class="ts" x="555" y="102" text-anchor="middle">Annotateurs humains</text>
<text class="ts" x="555" y="118" text-anchor="middle">"laquelle est meilleure ?"</text>

<!-- Final model -->
<rect x="200" y="190" width="280" height="60" rx="10" fill="#FAECE7" stroke="#993C1D" stroke-width="1.2"/>
<text class="th" x="340" y="215" text-anchor="middle">Modèle assistant final</text>
<text class="ts" x="340" y="235" text-anchor="middle">ChatGPT, Claude, Gemini...</text>

<line x1="335" y1="130" x2="310" y2="188" stroke="#888" stroke-width="1" marker-end="url(#arr)"/>
<line x1="555" y1="130" x2="400" y2="188" stroke="#888" stroke-width="1" marker-end="url(#arr)"/>

<!-- One foundation many apps -->
<text class="th" x="340" y="286" text-anchor="middle">Un seul pré-entraînement → de nombreuses spécialisations</text>
<rect x="40" y="298" width="130" height="18" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".5"/><text class="ts" x="105" y="311" text-anchor="middle">Assistant général</text>
<rect x="185" y="298" width="100" height="18" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".5"/><text class="ts" x="235" y="311" text-anchor="middle">Médecine</text>
<rect x="300" y="298" width="80" height="18" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".5"/><text class="ts" x="340" y="311" text-anchor="middle">Code</text>
<rect x="395" y="298" width="80" height="18" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".5"/><text class="ts" x="435" y="311" text-anchor="middle">Juridique</text>
<rect x="490" y="298" width="150" height="18" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".5"/><text class="ts" x="565" y="311" text-anchor="middle">Domaine spécialisé...</text>
</svg>
''')

## 3.1 — LoRA : fine-tuning efficace sans tout réentraîner

Au lieu de modifier tous les paramètres du modèle, LoRA n'entraîne qu'une **petite matrice de rang faible**
injectée dans les couches d'attention. Le modèle original reste gelé.

```
W_original (768×768) → figé
+
A (768×r) × B (r×768)  ← seules ces matrices s'entraînent, r << 768
```

**Résultat :** fine-tuner un modèle de 7B paramètres sur un seul GPU en quelques heures.


---
# Partie 4 — Inférence : génération token par token

À l'inférence, le modèle génère un token à la fois de manière **autorégressive** :
chaque prédiction devient l'entrée du pas suivant.


In [ ]:
from IPython.display import HTML
HTML('''
<svg width="100%" viewBox="0 0 680 320" xmlns="http://www.w3.org/2000/svg">
<style>svg{font-family:system-ui,sans-serif;} .ts{font-size:12px;fill:#555;} .th{font-size:13px;font-weight:600;fill:#1a1a1a;}</style>
<defs><marker id="arr" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/></marker></defs>

<text class="ts" x="340" y="20" text-anchor="middle" fill="#888">Prompt : "Le chat"</text>

<!-- Step 1 -->
<text class="th" x="40" y="48">Étape 1</text>
<rect x="130" y="34" width="56" height="28" rx="5" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="158" y="53" text-anchor="middle">"Le"</text>
<rect x="194" y="34" width="60" height="28" rx="5" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="224" y="53" text-anchor="middle">"chat"</text>
<line x1="257" y1="48" x2="282" y2="48" stroke="#888" stroke-width="1" marker-end="url(#arr)"/>
<rect x="284" y="34" width="70" height="28" rx="6" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1"/><text class="ts" x="319" y="53" text-anchor="middle">Modèle</text>
<line x1="357" y1="48" x2="382" y2="48" stroke="#888" stroke-width="1" marker-end="url(#arr)"/>
<rect x="384" y="34" width="64" height="28" rx="5" fill="#FAECE7" stroke="#993C1D" stroke-width="1"/><text class="ts" x="416" y="53" text-anchor="middle">"dort"</text>

<!-- Step 2 -->
<text class="th" x="40" y="108">Étape 2</text>
<rect x="130" y="94" width="56" height="28" rx="5" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="158" y="113" text-anchor="middle">"Le"</text>
<rect x="194" y="94" width="60" height="28" rx="5" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="224" y="113" text-anchor="middle">"chat"</text>
<rect x="262" y="94" width="56" height="28" rx="5" fill="#FAECE7" stroke="#993C1D" stroke-width=".8"/><text class="ts" x="290" y="113" text-anchor="middle">"dort"</text>
<line x1="322" y1="108" x2="342" y2="108" stroke="#888" stroke-width="1" marker-end="url(#arr)"/>
<rect x="344" y="94" width="70" height="28" rx="6" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1"/><text class="ts" x="379" y="113" text-anchor="middle">Modèle</text>
<line x1="417" y1="108" x2="437" y2="108" stroke="#888" stroke-width="1" marker-end="url(#arr)"/>
<rect x="439" y="94" width="52" height="28" rx="5" fill="#FAECE7" stroke="#993C1D" stroke-width="1"/><text class="ts" x="465" y="113" text-anchor="middle">"sur"</text>

<!-- Step 3 -->
<text class="th" x="40" y="168">Étape 3</text>
<rect x="130" y="154" width="56" height="28" rx="5" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="158" y="173" text-anchor="middle">"Le"</text>
<rect x="194" y="154" width="60" height="28" rx="5" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="224" y="173" text-anchor="middle">"chat"</text>
<rect x="262" y="154" width="56" height="28" rx="5" fill="#FAECE7" stroke="#993C1D" stroke-width=".8"/><text class="ts" x="290" y="173" text-anchor="middle">"dort"</text>
<rect x="326" y="154" width="52" height="28" rx="5" fill="#FAECE7" stroke="#993C1D" stroke-width=".8"/><text class="ts" x="352" y="173" text-anchor="middle">"sur"</text>
<line x1="382" y1="168" x2="402" y2="168" stroke="#888" stroke-width="1" marker-end="url(#arr)"/>
<rect x="404" y="154" width="70" height="28" rx="6" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1"/><text class="ts" x="439" y="173" text-anchor="middle">Modèle</text>
<line x1="477" y1="168" x2="497" y2="168" stroke="#888" stroke-width="1" marker-end="url(#arr)"/>
<rect x="499" y="154" width="44" height="28" rx="5" fill="#FAECE7" stroke="#993C1D" stroke-width="1"/><text class="ts" x="521" y="173" text-anchor="middle">"le"</text>

<rect x="40" y="210" width="600" height="40" rx="8" fill="#F1EFE8" stroke="#5F5E5A" stroke-width=".5"/>
<text class="ts" x="340" y="226" text-anchor="middle">La génération s'arrête quand le modèle produit [END], atteint max_new_tokens,</text>
<text class="ts" x="340" y="242" text-anchor="middle">ou dépasse la fenêtre de contexte maximale (1024 pour GPT-2, 200k pour Claude).</text>

<text class="ts" x="340" y="290" text-anchor="middle" fill="#888">Toute la conversation précédente est passée au modèle à chaque tour.</text>
<text class="ts" x="340" y="308" text-anchor="middle" fill="#888">Le KV Cache évite de tout recalculer — seul le nouveau token est traité.</text>
</svg>
''')

## 4.1 — Stratégies de décodage : que faire des 50 257 probabilités ?

Une fois les probabilités calculées, plusieurs stratégies permettent de choisir le token suivant.
Prendre systématiquement le maximum (greedy) est simple mais produit des textes répétitifs.

| Stratégie | Principe | Avantage | Inconvénient |
|---|---|---|---|
| **Greedy** | Token de proba max | Rapide, déterministe | Répétitif |
| **Top-k** | Tirage parmi les k meilleurs | Plus varié | k fixe, parfois incohérent |
| **Top-p (nucleus)** | Tirage parmi les tokens couvrant p% de proba | Adaptatif, naturel | Standard actuel |


In [ ]:
from IPython.display import HTML
HTML('''
<svg width="100%" viewBox="0 0 680 240" xmlns="http://www.w3.org/2000/svg">
<style>svg{font-family:system-ui,sans-serif;} .ts{font-size:12px;fill:#555;} .th{font-size:13px;font-weight:600;fill:#1a1a1a;}</style>

<text class="th" x="40" y="24">Temperature + Top-p : pipeline de sélection</text>
<text class="ts" x="40" y="42">Logits bruts → ÷ Temperature → Softmax → Top-p → tirage</text>

<!-- Logits -->
<text class="ts" x="40" y="70">Logits bruts</text>
<rect x="40" y="78" width="52" height="70" rx="4" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="66" y="118" text-anchor="middle">8.1</text><text class="ts" x="66" y="138" text-anchor="middle">"dort"</text>
<rect x="100" y="100" width="52" height="48" rx="4" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="126" y="130" text-anchor="middle">5.2</text><text class="ts" x="126" y="148" text-anchor="middle" font-size="10">"ronronne"</text>
<rect x="160" y="116" width="52" height="32" rx="4" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="186" y="138" text-anchor="middle">3.8</text>
<rect x="220" y="138" width="52" height="10" rx="2" fill="#E6F1FB" stroke="#185FA5" stroke-width=".5"/>
<rect x="280" y="143" width="52" height="5" rx="2" fill="#E6F1FB" stroke="#185FA5" stroke-width=".5"/>

<!-- Arrow + Temperature -->
<text class="ts" x="352" y="100">÷ Temperature</text>
<text class="ts" x="352" y="118" fill="#BA7517">T&lt;1 → pique</text>
<text class="ts" x="352" y="136" fill="#993C1D">T&gt;1 → aplatit</text>

<!-- Arrow + Top-p -->
<text class="ts" x="480" y="100">Top-p = 0.9</text>
<rect x="480" y="108" width="52" height="28" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".8"/><text class="ts" x="506" y="127" text-anchor="middle">0.65</text>
<rect x="540" y="108" width="52" height="28" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".8"/><text class="ts" x="566" y="127" text-anchor="middle">0.20</text>
<rect x="600" y="108" width="52" height="28" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".8"/><text class="ts" x="626" y="127" text-anchor="middle">0.09</text>
<text class="ts" x="506" y="148" text-anchor="middle" fill="#0F6E56">cumul: .65</text>
<text class="ts" x="566" y="148" text-anchor="middle" fill="#0F6E56">.85</text>
<text class="ts" x="626" y="148" text-anchor="middle" fill="#0F6E56">.94 ✓</text>
<line x1="540" y1="104" x2="540" y2="155" stroke="#BA7517" stroke-width="1.5" stroke-dasharray="4 3"/>
<text class="ts" x="540" y="170" text-anchor="middle" fill="#BA7517">seuil p=0.9</text>

<rect x="40" y="195" width="600" height="36" rx="6" fill="#F1EFE8" stroke="#5F5E5A" stroke-width=".5"/>
<text class="ts" x="340" y="209" text-anchor="middle">Temperature façonne la distribution — Top-p découpe jusqu'où on pioche.</text>
<text class="ts" x="340" y="225" text-anchor="middle">C'est pourquoi deux réponses à la même question ne sont jamais identiques.</text>
</svg>
''')

---
# Partie 5 — Infrastructure & exploitation en production

Servir des LLMs à grande échelle est un défi d'ingénierie majeur.
Le KV Cache, la gestion mémoire GPU, et l'orchestration des requêtes sont les trois problèmes centraux.


## 5.1 — KV Cache : éviter de tout recalculer

Dans le mécanisme d'attention, chaque token produit une clé (K) et une valeur (V).
Ces vecteurs ne changent jamais pour les tokens déjà générés — on peut les mettre en cache.

| | Sans KV Cache | Avec KV Cache |
|---|---|---|
| Étape N | Recalcule K,V pour les N tokens | Réutilise le cache, calcule seulement le nouveau |
| Complexité | O(N²) | O(N) |
| Mémoire GPU | Faible | Croît avec la longueur de séquence |


In [ ]:
from IPython.display import HTML
HTML('''
<svg width="100%" viewBox="0 0 680 260" xmlns="http://www.w3.org/2000/svg">
<style>svg{font-family:system-ui,sans-serif;} .ts{font-size:12px;fill:#555;} .th{font-size:13px;font-weight:600;fill:#1a1a1a;}</style>
<defs><marker id="arr" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/></marker></defs>

<text class="th" x="160" y="24">Sans KV Cache — étape 3</text>
<text class="th" x="500" y="24">Avec KV Cache — étape 3</text>

<!-- Sans -->
<rect x="40" y="36" width="56" height="28" rx="5" fill="#F1EFE8" stroke="#5F5E5A" stroke-width=".5"/><text class="ts" x="68" y="55" text-anchor="middle">"Le"</text>
<text class="ts" x="68" y="78" text-anchor="middle" fill="#A32D2D">→K,V</text>
<rect x="104" y="36" width="60" height="28" rx="5" fill="#F1EFE8" stroke="#5F5E5A" stroke-width=".5"/><text class="ts" x="134" y="55" text-anchor="middle">"chat"</text>
<text class="ts" x="134" y="78" text-anchor="middle" fill="#A32D2D">→K,V</text>
<rect x="172" y="36" width="60" height="28" rx="5" fill="#FAEEDA" stroke="#BA7517" stroke-width="1"/><text class="ts" x="202" y="55" text-anchor="middle">"dort"</text>
<text class="ts" x="202" y="78" text-anchor="middle" fill="#A32D2D">→K,V</text>

<rect x="40" y="90" width="192" height="24" rx="4" fill="#FCEBEB" stroke="#A32D2D" stroke-width=".5"/>
<text class="ts" x="136" y="107" text-anchor="middle" fill="#A32D2D">3 calculs — 2 déjà faits !</text>

<!-- Avec -->
<rect x="360" y="36" width="56" height="28" rx="5" fill="#F1EFE8" stroke="#5F5E5A" stroke-width=".5"/><text class="ts" x="388" y="55" text-anchor="middle">"Le"</text>
<text class="ts" x="388" y="78" text-anchor="middle" fill="#0F6E56">✅ caché</text>
<rect x="424" y="36" width="60" height="28" rx="5" fill="#F1EFE8" stroke="#5F5E5A" stroke-width=".5"/><text class="ts" x="454" y="55" text-anchor="middle">"chat"</text>
<text class="ts" x="454" y="78" text-anchor="middle" fill="#0F6E56">✅ caché</text>
<rect x="492" y="36" width="60" height="28" rx="5" fill="#FAEEDA" stroke="#BA7517" stroke-width="1"/><text class="ts" x="522" y="55" text-anchor="middle">"dort"</text>
<text class="ts" x="522" y="78" text-anchor="middle" fill="#185FA5">🆕 calculé</text>

<rect x="360" y="90" width="192" height="24" rx="4" fill="#EAF3DE" stroke="#3B6D11" stroke-width=".5"/>
<text class="ts" x="456" y="107" text-anchor="middle" fill="#3B6D11">1 seul calcul ✅</text>

<!-- Cache en VRAM -->
<rect x="40" y="134" width="600" height="80" rx="8" fill="#F1EFE8" stroke="#5F5E5A" stroke-width=".8"/>
<text class="th" x="340" y="156" text-anchor="middle">KV Cache en mémoire GPU (VRAM)</text>
<rect x="60" y="168" width="300" height="22" rx="4" fill="#185FA5" opacity=".7"/>
<text class="ts" x="210" y="184" text-anchor="middle" fill="white">Poids du modèle (fixe)</text>
<rect x="368" y="168" width="160" height="22" rx="4" fill="#0F6E56" opacity=".7"/>
<text class="ts" x="448" y="184" text-anchor="middle" fill="white">KV Cache (↑ grandit)</text>
<rect x="536" y="168" width="84" height="22" rx="4" fill="#BA7517" opacity=".7"/>
<text class="ts" x="578" y="184" text-anchor="middle" fill="white">Activ.</text>

<text class="ts" x="340" y="228" text-anchor="middle" fill="#888">GPT-2 : ~36 Mo pour 1024 tokens. Modèle 70B sur 128k tokens : ~160 Go.</text>
<text class="ts" x="340" y="246" text-anchor="middle" fill="#888">Cache par conversation — pas partageable entre utilisateurs (sauf prompt caching sur préfixe commun).</text>
</svg>
''')

## 5.2 — PagedAttention : gestion mémoire inspirée des OS

Le problème : allouer un bloc contigu par conversation gaspille 60-80% de la VRAM.
La solution : **pages de taille fixe** allouées à la demande, exactement comme la mémoire virtuelle d'un OS.

> *Paper de référence : "Efficient Memory Management for Large Language Model Serving with PagedAttention" — SOSP'23 (UC Berkeley)*
> arxiv.org/abs/2309.06180 — améliore le débit de **2-4×** par rapport aux systèmes précédents.


In [ ]:
from IPython.display import HTML
HTML('''
<svg width="100%" viewBox="0 0 680 220" xmlns="http://www.w3.org/2000/svg">
<style>svg{font-family:system-ui,sans-serif;} .ts{font-size:12px;fill:#555;} .th{font-size:13px;font-weight:600;fill:#1a1a1a;}</style>

<text class="th" x="160" y="24">Sans PagedAttention</text>
<text class="th" x="500" y="24">Avec PagedAttention (vLLM)</text>

<!-- Sans -->
<rect x="40" y="36" width="240" height="100" rx="6" fill="none" stroke="#5F5E5A" stroke-width=".8"/>
<rect x="44" y="40" width="110" height="44" rx="4" fill="#E6F1FB" stroke="#185FA5" stroke-width=".5"/><text class="ts" x="99" y="67" text-anchor="middle">User A — 3 tok</text>
<rect x="158" y="40" width="118" height="44" rx="4" fill="#ddd" opacity=".5" stroke="#ccc" stroke-width=".5"/><text class="ts" x="217" y="67" text-anchor="middle" fill="#aaa">réservé vide</text>
<rect x="44" y="90" width="232" height="42" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".5"/><text class="ts" x="160" y="116" text-anchor="middle">User B — 8 tok (plein)</text>
<rect x="44" y="148" width="160" height="20" rx="4" fill="#FCEBEB" stroke="#A32D2D" stroke-width=".5"/>
<text class="ts" x="124" y="163" text-anchor="middle" fill="#A32D2D">❌ fragmentation — 30% gaspillé</text>

<!-- Avec -->
<rect x="360" y="36" width="280" height="100" rx="6" fill="none" stroke="#5F5E5A" stroke-width=".8"/>
<rect x="364" y="40" width="60" height="44" rx="4" fill="#E6F1FB" stroke="#185FA5" stroke-width=".5"/><text class="ts" x="394" y="67" text-anchor="middle">A p1</text>
<rect x="430" y="40" width="60" height="44" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".5"/><text class="ts" x="460" y="67" text-anchor="middle">B p1</text>
<rect x="496" y="40" width="60" height="44" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".5"/><text class="ts" x="526" y="67" text-anchor="middle">B p2</text>
<rect x="562" y="40" width="60" height="44" rx="4" fill="#E6F1FB" stroke="#185FA5" stroke-width=".5"/><text class="ts" x="592" y="67" text-anchor="middle">A p2</text>
<rect x="364" y="90" width="60" height="42" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".5"/><text class="ts" x="394" y="116" text-anchor="middle">B p3</text>
<rect x="430" y="90" width="60" height="42" rx="4" fill="#FAEEDA" stroke="#BA7517" stroke-width=".5"/><text class="ts" x="460" y="116" text-anchor="middle">C p1</text>
<rect x="496" y="90" width="60" height="42" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".5"/><text class="ts" x="526" y="116" text-anchor="middle">B p4</text>
<rect x="562" y="90" width="60" height="42" rx="4" fill="#FAEEDA" stroke="#BA7517" stroke-width=".5"/><text class="ts" x="592" y="116" text-anchor="middle">C p2</text>
<rect x="360" y="148" width="200" height="20" rx="4" fill="#EAF3DE" stroke="#3B6D11" stroke-width=".5"/>
<text class="ts" x="460" y="163" text-anchor="middle" fill="#3B6D11">✅ &lt;4% gaspillage</text>

<rect x="40" y="185" width="600" height="30" rx="6" fill="#F1EFE8" stroke="#5F5E5A" stroke-width=".5"/>
<text class="ts" x="340" y="205" text-anchor="middle">Une table de pages mappe les blocs logiques de chaque user sur des pages physiques VRAM non-contiguës.</text>
</svg>
''')

## 5.3 — Continuous batching & architecture globale

Le continuous batching insère dynamiquement de nouvelles requêtes dans le batch dès qu'un slot se libère.
Le GPU n'est jamais idle entre deux requêtes.


In [ ]:
from IPython.display import HTML
HTML('''
<svg width="100%" viewBox="0 0 680 300" xmlns="http://www.w3.org/2000/svg">
<style>svg{font-family:system-ui,sans-serif;} .ts{font-size:12px;fill:#555;} .th{font-size:13px;font-weight:600;fill:#1a1a1a;}</style>
<defs><marker id="arr" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/></marker></defs>

<!-- Timeline -->
<text class="th" x="340" y="24" text-anchor="middle">Continuous batching — occupation GPU dans le temps</text>

<text class="ts" x="30" y="58" text-anchor="end">Slot 1</text>
<rect x="40" y="44" width="240" height="26" rx="4" fill="#E6F1FB" stroke="#185FA5" stroke-width=".8"/><text class="ts" x="160" y="62" text-anchor="middle">User A (court)</text>
<rect x="284" y="44" width="356" height="26" rx="4" fill="#FAEEDA" stroke="#BA7517" stroke-width=".8"/><text class="ts" x="462" y="62" text-anchor="middle">User D — inséré dès que A termine ✅</text>

<text class="ts" x="30" y="98" text-anchor="end">Slot 2</text>
<rect x="40" y="84" width="600" height="26" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".8"/><text class="ts" x="340" y="102" text-anchor="middle">User B (long)</text>

<text class="ts" x="30" y="138" text-anchor="end">Slot 3</text>
<rect x="40" y="124" width="360" height="26" rx="4" fill="#FAECE7" stroke="#993C1D" stroke-width=".8"/><text class="ts" x="220" y="142" text-anchor="middle">User C (moyen)</text>
<rect x="404" y="124" width="236" height="26" rx="4" fill="#FAEEDA" stroke="#BA7517" stroke-width=".8"/><text class="ts" x="522" y="142" text-anchor="middle">User E — inséré ✅</text>

<text class="ts" x="30" y="178" text-anchor="end">GPU</text>
<rect x="40" y="164" width="600" height="26" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1.2"/>
<text class="ts" x="340" y="182" text-anchor="middle" font-weight="600">✅ GPU occupé en permanence — 0% idle</text>

<!-- Stack -->
<line x1="40" y1="210" x2="640" y2="210" stroke="#ddd" stroke-width=".8"/>
<text class="th" x="340" y="232" text-anchor="middle">Stack en production</text>
<rect x="40" y="242" width="600" height="22" rx="4" fill="#F1EFE8" stroke="#5F5E5A" stroke-width=".5"/>
<text class="ts" x="340" y="258" text-anchor="middle">Kubernetes — cycle de vie des pods GPU, autoscaling, health checks</text>
<rect x="40" y="268" width="600" height="22" rx="4" fill="#E1F5EE" stroke="#0F6E56" stroke-width=".5"/>
<text class="ts" x="340" y="284" text-anchor="middle">vLLM / TGI — PagedAttention, continuous batching, gestion KV cache, swap CPU↔GPU</text>
</svg>
''')

---
# Conclusion — Vue d'ensemble

```
Corpus (milliards de tokens)
        ↓
  Pré-entraînement          ← BPE, Transformer, next token prediction
        ↓
  Modèle fondationnel       ← sait continuer du texte, ne sait pas répondre
        ↓
  Fine-tuning (SFT)         ← 10k-100k exemples question→réponse
        ↓
  RLHF                      ← annotateurs humains, reward model
        ↓
  Modèle assistant          ← Claude, GPT-4, Gemini...
        ↓
  Inférence                 ← génération autorégressive, KV Cache
        ↓
  Production                ← vLLM, PagedAttention, Kubernetes
```

## Points clés à retenir

- **BPE** : tokenisation par fusion itérative des paires fréquentes
- **Masked attention** : masque triangulaire — artifice d'entraînement, naturel à l'inférence
- **Pré-entraînement** : très coûteux (~100M$), fait une seule fois par les grands labs
- **Fine-tuning** : bien moins coûteux, accessible avec LoRA sur un seul GPU
- **KV Cache** : réduit la complexité de O(N²) à O(N), stocké en VRAM par conversation
- **PagedAttention** : gestion paginée du KV cache, réduit le gaspillage de 80% à <4%
- **Top-p + Temperature** : contrôlent le compromis créativité/cohérence à l'inférence
- **Fenêtre de contexte** : toute la conversation passe à chaque tour — le KV cache évite le recalcul
